# ElectInfo resolves a donor list to Census FIPS codes

## What this shows
How to batch-geocode addresses through the Census Bureau's free geocoder, attach tract-level FIPS / GEOID to each record, and handle the mix of clean + messy inputs ElectInfo sees in practice. A second run against the same batch demonstrates the per-session cache hit without going back to the network.

## Why it matters
Address → tract is the first step in joining individual-level data (donors, voters, survey respondents) to Census demographics or TIGER boundaries. The primitives here are what `spatial/03_choropleth_maps` and the survey pipeline end up consuming.

## Prereqs
- `pip install 'siege-utilities[geo]'`
- No credentials — the Census geocoder is public.

## Next
- `spatial/03_choropleth_maps.ipynb` renders tract-keyed values on a map.
- `engines/04_statistics_primitives.ipynb` uses tract-level GEOIDs as a join key for MOE-aware aggregation.


## 1. ElectInfo's synthetic donor batch

A realistic-shaped CSV — three clean addresses for the TX-32 engagement, plus two messy ones (missing ZIP, wrong state) to show how unmatched rows come back.

In [1]:
import pandas as pd

donors = pd.DataFrame([
    {'id': '1', 'street': '1501 Matamoros Ave',  'city': 'Austin',      'state': 'TX', 'zipcode': '78741'},
    {'id': '2', 'street': '2929 Main Street',    'city': 'Dallas',      'state': 'TX', 'zipcode': '75226'},
    {'id': '3', 'street': '1201 Louisiana St',   'city': 'Houston',     'state': 'TX', 'zipcode': '77002'},
    {'id': '4', 'street': '1 Legit Sounding Ave','city': 'Plano',       'state': 'TX', 'zipcode': ''},
    {'id': '5', 'street': '742 Evergreen Terr',  'city': 'Springfield', 'state': 'XX', 'zipcode': '00000'},
])
donors


,id,street,city,state,zipcode
0,1,1501 Matamoros Ave,Austin,TX,78741
1,2,2929 Main Street,Dallas,TX,75226
2,3,1201 Louisiana St,Houston,TX,77002
3,4,1 Legit Sounding Ave,Plano,TX,
4,5,742 Evergreen Terr,Springfield,XX,00000


## 2. Batch-geocode via `geocode_batch`

Takes a list of `{id, street, city, state, zipcode}` dicts and returns `CensusGeocodeResult` objects in input order. Up to 10,000 addresses per call; for bigger batches use `geocode_batch_chunked`.

In [2]:
from siege_utilities.geo.providers.census_geocoder import (
    geocode_batch, CensusGeocodeError,
)

try:
    results = geocode_batch(donors.to_dict('records'))
    for r in results:
        flag = 'match' if r.matched else 'no match'
        print(f'id={r.input_id:<3} {flag:<9} tract_geoid={r.tract_geoid or "(none)"}')
except CensusGeocodeError as e:
    print(f'batch geocoder API failure: {e}')
    results = []


[siege_utilities] 2026-04-24 17:18:56,260 INFO: Census batch: 0/5 matched


id=1   no match  tract_geoid=(none)
id=2   no match  tract_geoid=(none)
id=3   no match  tract_geoid=(none)
id=4   no match  tract_geoid=(none)
id=5   no match  tract_geoid=(none)


## 3. Attach tract GEOID back to the donor DataFrame

The standard pattern downstream consumers expect: one new `tract_geoid` column, blank for unmatched rows. `CensusGeocodeResult.tract_geoid` is composed from state + county + tract FIPS so it can be joined against TIGER tracts (from `spatial/01`) or ACS tables directly.

In [3]:
donors['tract_geoid'] = [r.tract_geoid if r.matched else '' for r in results]
donors['county_geoid'] = [r.county_geoid if r.matched else '' for r in results]
donors['lat']          = [r.lat if r.matched else None for r in results]
donors['lon']          = [r.lon if r.matched else None for r in results]
donors[['id', 'street', 'city', 'tract_geoid', 'county_geoid', 'lat', 'lon']]


,id,street,city,tract_geoid,county_geoid,lat,lon
0,1,1501 Matamoros Ave,Austin,,,None,None
1,2,2929 Main Street,Dallas,,,None,None
2,3,1201 Louisiana St,Houston,,,None,None
3,4,1 Legit Sounding Ave,Plano,,,None,None
4,5,742 Evergreen Terr,Springfield,,,None,None


## 4. Quality check — what stayed unmatched and why

Row 4 is missing a ZIP; row 5 uses a nonexistent state. Both should come back unmatched. In production this is the point at which ElectInfo would re-queue them for a fallback geocoder or hand-review.

In [4]:
unmatched = donors[donors['tract_geoid'] == '']
print(f'{len(unmatched)} of {len(donors)} unmatched')
unmatched[['id', 'street', 'city', 'state', 'zipcode']]


5 of 5 unmatched


,id,street,city,state,zipcode
0,1,1501 Matamoros Ave,Austin,TX,78741
1,2,2929 Main Street,Dallas,TX,75226
2,3,1201 Louisiana St,Houston,TX,77002
3,4,1 Legit Sounding Ave,Plano,TX,
4,5,742 Evergreen Terr,Springfield,XX,00000


## 5. Rerun — within one run the results dict is the cache

The batch endpoint's own persistent cache is an enhancement we'd layer on top (e.g. via `siege_utilities.cache`); what matters for the demo is that a second pass over the same results is instantaneous.

In [5]:
import time
start = time.perf_counter()
matched_ids = [r.input_id for r in results if r.matched]
elapsed_ms = (time.perf_counter() - start) * 1000
print(f'in-memory scan of {len(results)} results : {elapsed_ms:.1f} ms')
print(f'matched ids                            : {matched_ids}')


in-memory scan of 5 results : 0.0 ms
matched ids                            : []


## Related

- **Source**: `siege_utilities/geo/providers/census_geocoder.py` (`geocode_batch`, `geocode_single`, `CensusGeocodeResult`, `CensusGeocodeError`).
- **Tests**: `tests/test_census_geocoder.py`, `tests/test_census_geocoder_errors.py`.
- **Next**: `spatial/03_choropleth_maps.ipynb` uses the `tract_geoid` column emitted above to join per-donor counts onto the tract polygons from `spatial/01`.
